# Backtest B3 — Quality + Momentum Long-Only

**Objetivo:** construir e testar uma estratégia multifatorial long-only para ações da B3 entre **2021 e 2026**, usando fatores de **Quality** e **Momentum**.

## Estratégia

Em cada rebalanceamento trimestral:

1. Seleciona as ações mais líquidas da B3.
2. Calcula fatores de Quality:
   - ROE alto;
   - Dívida Líquida/EBITDA baixa;
   - opcionalmente Earnings Yield, se P/L estiver disponível.
3. Calcula Momentum:
   - retorno acumulado de 12 meses excluindo o último mês, ou seja, momentum 12-1.
4. Cria um score multifatorial.
5. Seleciona as melhores ações.
6. Monta carteira long-only.
7. Aplica custos de transação.
8. Compara contra benchmarks, como Ibovespa, IBrX e CDI, se disponíveis.

---

## Arquivos esperados

Coloque os arquivos na mesma pasta deste notebook.

### `prices_b3.csv`

Colunas mínimas:

```text
date,ticker,adjusted_close,volume_financial
```

### `fundamentals_b3.csv`

Colunas mínimas:

```text
report_date,ticker,sector,roe,net_debt_ebitda
```

Coluna opcional:

```text
pe_ratio
```

### `benchmarks_b3.csv`, opcional

Colunas esperadas:

```text
date,ibov_return,ibrx_return,cdi_return
```

Caso você não tenha benchmark ainda, o notebook roda sem ele.

---

## Observação importante

Se os nomes das colunas exportadas do Economática vierem diferentes, ajuste no bloco **Mapeamento de colunas**.

In [ ]:
# ============================================================
# 0. Instalação, se necessário
# ============================================================

# Rode esta célula se estiver faltando alguma biblioteca:
# !pip install pandas numpy matplotlib openpyxl

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Dict, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Bibliotecas carregadas.")

In [ ]:
# ============================================================
# 2. Configuração do backtest
# ============================================================

@dataclass
class BacktestConfig:
    # Período pedido
    start_date: str = "2021-01-01"
    end_date: str = "2026-12-31"

    # Universo e carteira
    n_universe: int = 100              # 100 ações mais líquidas
    n_stocks: int = 20                 # carteira com 20 ações

    # Rebalanceamento
    rebalance_frequency: str = "Q"     # Q = trimestral, M = mensal
    reporting_lag_months: int = 3      # lag para evitar look-ahead bias

    # Janelas
    lookback_days: int = 252           # 12 meses de pregões
    min_lookback_days: int = 126       # mínimo de dados para estimar fatores
    momentum_skip_days: int = 21       # exclui o mês mais recente no momentum

    # Custos
    transaction_cost: float = 0.002    # 20 bps por turnover
    risk_free_daily: float = 0.0       # pode ser substituído por CDI diário

    # Controles
    sector_neutral: bool = False
    use_sector_limit: bool = True
    max_sector_weight: float = 0.30
    use_correlation_filter: bool = True
    corr_threshold: float = 0.85

    # Ponderação
    weighting_method: str = "equal_weight"  # "equal_weight" ou "inverse_vol"

    # Filtros
    exclude_negative_roe: bool = False
    exclude_financials: bool = False
    use_value_if_available: bool = False    # deixa False para focar Quality + Momentum

    # Arquivos
    prices_path: str = "prices_b3.csv"
    fundamentals_path: str = "fundamentals_b3.csv"
    benchmarks_path: str = "benchmarks_b3.csv"

    # Saída
    output_dir: str = "output_b3_quality_momentum"


config = BacktestConfig()

config

# 3. Modo de dados

Por padrão, o notebook tenta carregar os CSVs reais do Economática.

Se os arquivos ainda não estiverem na pasta, ele pode gerar uma base **simulada** apenas para testar se o pipeline funciona.

Para o trabalho final, deixe:

```python
USE_DEMO_DATA = False
```

e coloque os arquivos reais na pasta.

In [ ]:
# ============================================================
# 3. Controle de dados
# ============================================================

USE_DEMO_DATA = False  # deixe True só para testar o pipeline sem Economática

DATA_DIR = Path(".")
OUTPUT_DIR = Path(config.output_dir)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Pasta atual: {Path.cwd()}")
print(f"Saída será salva em: {OUTPUT_DIR.resolve()}")

In [ ]:
# ============================================================
# 4. Funções auxiliares gerais
# ============================================================

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df


def read_file(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    if path.suffix.lower() == ".csv":
        # sep=None tenta detectar ; ou ,
        return pd.read_csv(path, sep=None, engine="python", decimal=".")
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Formato não suportado: {path.suffix}")


def fix_decimal_columns(df: pd.DataFrame, numeric_cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            # Trata número brasileiro "1.234,56" quando vier como string
            if df[col].dtype == "object":
                s = df[col].astype(str).str.strip()
                s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
                df[col] = pd.to_numeric(s, errors="coerce")
            else:
                df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def winsorize_series(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    s = s.copy()
    if s.dropna().empty:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def zscore_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    std = s.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / std


def compute_drawdown(returns: pd.Series) -> pd.Series:
    cumulative = (1 + returns.fillna(0)).cumprod()
    running_max = cumulative.cummax()
    return cumulative / running_max - 1


def annualize_return(returns: pd.Series, periods_per_year: int = 252) -> float:
    r = returns.dropna()
    if len(r) == 0:
        return np.nan
    total_return = (1 + r).prod() - 1
    return (1 + total_return) ** (periods_per_year / len(r)) - 1


def sharpe_ratio(returns: pd.Series, risk_free_daily: float = 0.0) -> float:
    r = returns.dropna()
    if len(r) == 0:
        return np.nan
    excess = r - risk_free_daily
    vol = excess.std(ddof=0)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return excess.mean() / vol * np.sqrt(252)


def sortino_ratio(returns: pd.Series, risk_free_daily: float = 0.0) -> float:
    r = returns.dropna()
    if len(r) == 0:
        return np.nan
    excess = r - risk_free_daily
    downside = excess[excess < 0]
    if len(downside) == 0:
        return np.nan
    downside_dev = np.sqrt((downside ** 2).mean())
    if downside_dev == 0 or np.isnan(downside_dev):
        return np.nan
    return excess.mean() / downside_dev * np.sqrt(252)


print("Funções auxiliares carregadas.")

# 5. Mapeamento de colunas

O Economática pode exportar nomes diferentes, como `Data`, `Ativo`, `Fechamento Ajustado`, etc.

Depois de carregar os dados, o notebook padroniza os nomes, mas você talvez precise editar os dicionários abaixo.

In [ ]:
# ============================================================
# 5. Mapeamento de colunas
# ============================================================

# Ajuste aqui se seu CSV vier com nomes diferentes.
# O lado esquerdo é o nome que veio no arquivo depois de padronizar.
# O lado direito é o nome que o backtest usa.

PRICE_COLUMN_MAP = {
    # exemplos:
    # "data": "date",
    # "ativo": "ticker",
    # "ticker": "ticker",
    # "preco_ajustado": "adjusted_close",
    # "fechamento_ajustado": "adjusted_close",
    # "volume_financeiro": "volume_financial",
}

FUND_COLUMN_MAP = {
    # exemplos:
    # "data": "report_date",
    # "data_do_balanco": "report_date",
    # "ativo": "ticker",
    # "setor": "sector",
    # "roe": "roe",
    # "divida_liquida_ebitda": "net_debt_ebitda",
    # "pl": "pe_ratio",
}

BENCH_COLUMN_MAP = {
    # exemplos:
    # "data": "date",
    # "ibovespa": "ibov_return",
    # "ibrx": "ibrx_return",
    # "cdi": "cdi_return",
}


def apply_column_map(df: pd.DataFrame, mapping: Dict[str, str]) -> pd.DataFrame:
    df = df.copy()
    existing = {k: v for k, v in mapping.items() if k in df.columns}
    return df.rename(columns=existing)

In [ ]:
# ============================================================
# 6. Geração de dados simulados para teste opcional
# ============================================================

def generate_demo_data(seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)

    dates = pd.bdate_range("2019-01-01", "2026-05-15")
    n_assets = 140

    sectors = [
        "Financeiro", "Energia", "Materiais", "Consumo", "Varejo",
        "Utilities", "Saude", "Tecnologia", "Industria", "Petróleo"
    ]

    tickers = [f"ACAO{i:03d}" for i in range(1, n_assets + 1)]
    sector_map = {t: rng.choice(sectors) for t in tickers}

    price_rows = []
    fund_rows = []

    for ticker in tickers:
        drift = rng.normal(0.00025, 0.00015)
        vol = rng.uniform(0.012, 0.035)
        start_price = rng.uniform(8, 80)

        returns = rng.normal(drift, vol, len(dates))
        prices = start_price * np.cumprod(1 + returns)
        volume_base = rng.lognormal(mean=15.0, sigma=1.0)

        for d, p, r in zip(dates, prices, returns):
            volume = max(1_000_000, volume_base * rng.lognormal(0, 0.35))
            price_rows.append([d, ticker, p, volume])

        quarter_dates = pd.date_range("2019-03-31", "2026-03-31", freq="Q")
        quality_latent = rng.normal(0, 1)

        for qd in quarter_dates:
            roe = np.clip(0.10 + 0.06 * quality_latent + rng.normal(0, 0.05), -0.20, 0.45)
            net_debt_ebitda = np.clip(2.2 - 0.6 * quality_latent + rng.normal(0, 1.0), -2.0, 8.0)
            pe_ratio = np.clip(14 + rng.normal(0, 6) - 2 * quality_latent, 3, 45)
            fund_rows.append([qd, ticker, sector_map[ticker], roe, net_debt_ebitda, pe_ratio])

    prices = pd.DataFrame(price_rows, columns=["date", "ticker", "adjusted_close", "volume_financial"])
    fundamentals = pd.DataFrame(fund_rows, columns=["report_date", "ticker", "sector", "roe", "net_debt_ebitda", "pe_ratio"])

    # Benchmark simulado
    bench = pd.DataFrame({"date": dates})
    bench["ibov_return"] = rng.normal(0.00022, 0.018, len(dates))
    bench["ibrx_return"] = rng.normal(0.00024, 0.017, len(dates))
    bench["cdi_return"] = 0.00045

    return prices, fundamentals, bench

In [ ]:
# ============================================================
# 7. Carregamento e validação dos dados
# ============================================================

def load_prices(path: str) -> pd.DataFrame:
    df = read_file(path)
    df = standardize_column_names(df)
    df = apply_column_map(df, PRICE_COLUMN_MAP)

    required = ["date", "ticker", "adjusted_close", "volume_financial"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print("Colunas encontradas em prices:")
        print(list(df.columns))
        raise ValueError(f"prices_b3 está sem colunas obrigatórias: {missing}")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

    df = fix_decimal_columns(df, ["adjusted_close", "volume_financial"])

    df = df.dropna(subset=["date", "ticker", "adjusted_close"])
    df = df[df["adjusted_close"] > 0]
    df = df.sort_values(["ticker", "date"])

    return df


def load_fundamentals(path: str) -> pd.DataFrame:
    df = read_file(path)
    df = standardize_column_names(df)
    df = apply_column_map(df, FUND_COLUMN_MAP)

    required = ["report_date", "ticker", "sector", "roe", "net_debt_ebitda"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print("Colunas encontradas em fundamentals:")
        print(list(df.columns))
        raise ValueError(f"fundamentals_b3 está sem colunas obrigatórias: {missing}")

    if "pe_ratio" not in df.columns:
        df["pe_ratio"] = np.nan

    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    df["sector"] = df["sector"].astype(str).str.strip()

    df = fix_decimal_columns(df, ["roe", "net_debt_ebitda", "pe_ratio"])

    # Se ROE vier em percentual tipo 18,5, converte para 0,185
    median_abs_roe = df["roe"].abs().median(skipna=True)
    if pd.notna(median_abs_roe) and median_abs_roe > 1:
        df["roe"] = df["roe"] / 100.0

    df = df.dropna(subset=["report_date", "ticker"])
    df = df.sort_values(["ticker", "report_date"])

    return df


def load_benchmarks(path: str) -> Optional[pd.DataFrame]:
    if not Path(path).exists():
        return None

    df = read_file(path)
    df = standardize_column_names(df)
    df = apply_column_map(df, BENCH_COLUMN_MAP)

    if "date" not in df.columns:
        print("Colunas encontradas em benchmarks:")
        print(list(df.columns))
        raise ValueError("benchmarks_b3 precisa ter coluna date.")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    for col in df.columns:
        if col != "date":
            df = fix_decimal_columns(df, [col])

    df = df.dropna(subset=["date"]).sort_values("date")
    return df


if USE_DEMO_DATA:
    prices, fundamentals, benchmarks = generate_demo_data()
    print("Usando dados simulados para testar o pipeline.")
else:
    prices_file = DATA_DIR / config.prices_path
    fundamentals_file = DATA_DIR / config.fundamentals_path
    benchmarks_file = DATA_DIR / config.benchmarks_path

    prices = load_prices(prices_file)
    fundamentals = load_fundamentals(fundamentals_file)
    benchmarks = load_benchmarks(benchmarks_file)

print("Dados carregados.")
print(f"Preços: {prices.shape}")
print(f"Fundamentos: {fundamentals.shape}")
print(f"Benchmarks: {None if benchmarks is None else benchmarks.shape}")

In [ ]:
# ============================================================
# 8. Diagnóstico dos dados
# ============================================================

def data_diagnostics(prices: pd.DataFrame, fundamentals: pd.DataFrame, benchmarks: Optional[pd.DataFrame]) -> None:
    print("=" * 70)
    print("DIAGNÓSTICO DOS DADOS")
    print("=" * 70)

    print("\nPREÇOS")
    print(f"Linhas: {len(prices):,}")
    print(f"Tickers: {prices['ticker'].nunique():,}")
    print(f"Período: {prices['date'].min().date()} até {prices['date'].max().date()}")
    display(prices.head())

    print("\nFUNDAMENTOS")
    print(f"Linhas: {len(fundamentals):,}")
    print(f"Tickers: {fundamentals['ticker'].nunique():,}")
    print(f"Período: {fundamentals['report_date'].min().date()} até {fundamentals['report_date'].max().date()}")
    display(fundamentals.head())

    common = set(prices["ticker"].unique()).intersection(set(fundamentals["ticker"].unique()))
    print(f"\nTickers em comum: {len(common):,}")

    if benchmarks is not None:
        print("\nBENCHMARKS")
        print(f"Linhas: {len(benchmarks):,}")
        print(f"Período: {benchmarks['date'].min().date()} até {benchmarks['date'].max().date()}")
        display(benchmarks.head())
    else:
        print("\nSem benchmark carregado.")

data_diagnostics(prices, fundamentals, benchmarks)

In [ ]:
# ============================================================
# 9. Preparação de preços e fundamentos
# ============================================================

def prepare_prices(prices: pd.DataFrame) -> pd.DataFrame:
    df = prices.copy()
    df = df.sort_values(["ticker", "date"])
    df["return"] = df.groupby("ticker")["adjusted_close"].pct_change()
    df = df.replace([np.inf, -np.inf], np.nan)
    return df


def prepare_fundamentals(fundamentals: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    df = fundamentals.copy()
    df["available_date"] = df["report_date"] + pd.DateOffset(months=config.reporting_lag_months)

    if "pe_ratio" in df.columns:
        df["earnings_yield"] = np.where(df["pe_ratio"] > 0, 1.0 / df["pe_ratio"], np.nan)
    else:
        df["earnings_yield"] = np.nan

    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.sort_values(["ticker", "available_date"])

    return df


prices_prepared = prepare_prices(prices)
fundamentals_prepared = prepare_fundamentals(fundamentals, config)

display(prices_prepared.head())
display(fundamentals_prepared.head())

In [ ]:
# ============================================================
# 10. Cálculo dos fatores de mercado: liquidez, vol, sortino, momentum
# ============================================================

def compute_market_factors(prices: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    df = prices.copy()
    df = df.sort_values(["ticker", "date"])

    all_parts = []

    for ticker, g in df.groupby("ticker", sort=False):
        g = g.sort_values("date").copy()

        g["adtv_12m"] = (
            g["volume_financial"]
            .rolling(config.lookback_days, min_periods=config.min_lookback_days)
            .mean()
        )

        g["vol_12m"] = (
            g["return"]
            .rolling(config.lookback_days, min_periods=config.min_lookback_days)
            .std()
            * np.sqrt(252)
        )

        g["sharpe_12m"] = (
            g["return"]
            .rolling(config.lookback_days, min_periods=config.min_lookback_days)
            .apply(lambda x: sharpe_ratio(pd.Series(x), config.risk_free_daily), raw=False)
        )

        g["sortino_12m"] = (
            g["return"]
            .rolling(config.lookback_days, min_periods=config.min_lookback_days)
            .apply(lambda x: sortino_ratio(pd.Series(x), config.risk_free_daily), raw=False)
        )

        # Momentum 12-1: preço de 21 pregões atrás / preço de 252 pregões atrás - 1
        shifted_price = g["adjusted_close"].shift(config.momentum_skip_days)
        past_price = g["adjusted_close"].shift(config.lookback_days)
        g["momentum_12_1"] = shifted_price / past_price - 1

        all_parts.append(g)

    out = pd.concat(all_parts, ignore_index=True)
    out = out.replace([np.inf, -np.inf], np.nan)

    return out


market_factors = compute_market_factors(prices_prepared, config)

display(market_factors.tail())
print(market_factors[["adtv_12m", "vol_12m", "sharpe_12m", "sortino_12m", "momentum_12_1"]].describe())

In [ ]:
# ============================================================
# 11. Datas de rebalanceamento e snapshots
# ============================================================

def get_rebalance_dates(prices: pd.DataFrame, config: BacktestConfig) -> pd.DatetimeIndex:
    dates = pd.Series(pd.to_datetime(prices["date"].unique())).sort_values()
    temp = pd.DataFrame({"date": dates})

    if config.rebalance_frequency.upper() == "Q":
        temp["period"] = temp["date"].dt.to_period("Q")
    elif config.rebalance_frequency.upper() == "M":
        temp["period"] = temp["date"].dt.to_period("M")
    else:
        raise ValueError("rebalance_frequency deve ser 'Q' ou 'M'.")

    rebalance_dates = temp.groupby("period")["date"].max().sort_values()
    return pd.DatetimeIndex(rebalance_dates.values)


def latest_market_snapshot(market_factors: pd.DataFrame, asof_date: pd.Timestamp) -> pd.DataFrame:
    available = market_factors[market_factors["date"] <= asof_date].copy()
    if available.empty:
        return pd.DataFrame()

    latest = (
        available
        .sort_values(["ticker", "date"])
        .groupby("ticker", as_index=False)
        .tail(1)
    )
    return latest


def latest_fundamentals_snapshot(fundamentals: pd.DataFrame, asof_date: pd.Timestamp) -> pd.DataFrame:
    available = fundamentals[fundamentals["available_date"] <= asof_date].copy()
    if available.empty:
        return pd.DataFrame()

    latest = (
        available
        .sort_values(["ticker", "available_date"])
        .groupby("ticker", as_index=False)
        .tail(1)
    )
    return latest


rebalance_dates = get_rebalance_dates(prices_prepared, config)
rebalance_dates = rebalance_dates[
    (rebalance_dates >= pd.Timestamp(config.start_date)) &
    (rebalance_dates <= pd.Timestamp(config.end_date))
]

print(f"Número de rebalanceamentos: {len(rebalance_dates)}")
print(rebalance_dates[:5])
print(rebalance_dates[-5:])

In [ ]:
# ============================================================
# 12. Score Quality + Momentum
# ============================================================

def select_liquid_universe(market_snapshot: pd.DataFrame, config: BacktestConfig) -> List[str]:
    df = market_snapshot.dropna(subset=["adtv_12m"]).copy()
    df = df.sort_values("adtv_12m", ascending=False).head(config.n_universe)
    return df["ticker"].tolist()


def apply_basic_filters(data: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    df = data.copy()

    required = [
        "roe",
        "net_debt_ebitda",
        "vol_12m",
        "sortino_12m",
        "momentum_12_1",
        "adtv_12m",
    ]

    if config.use_value_if_available:
        required.append("earnings_yield")

    df = df.dropna(subset=required)
    df = df[df["vol_12m"] > 0]
    df = df[df["adtv_12m"] > 0]

    if config.exclude_negative_roe:
        df = df[df["roe"] > 0]

    if config.exclude_financials:
        financial_keywords = ["financ", "banco", "seguro", "insurance", "financial"]
        sector_lower = df["sector"].astype(str).str.lower()
        mask_financial = sector_lower.apply(lambda x: any(k in x for k in financial_keywords))
        df = df[~mask_financial]

    return df


def compute_quality_momentum_score(data: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    df = data.copy()

    factor_cols = [
        "roe",
        "net_debt_ebitda",
        "momentum_12_1",
        "vol_12m",
        "sortino_12m",
    ]

    if config.use_value_if_available and "earnings_yield" in df.columns:
        factor_cols.append("earnings_yield")

    for col in factor_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

    # Winsorização e z-score
    for col in factor_cols:
        if config.sector_neutral and "sector" in df.columns:
            df[col + "_w"] = df.groupby("sector")[col].transform(winsorize_series)
            df[col + "_z"] = df.groupby("sector")[col + "_w"].transform(zscore_series)
        else:
            df[col + "_w"] = winsorize_series(df[col])
            df[col + "_z"] = zscore_series(df[col + "_w"])

    # Quality:
    # ROE alto é positivo; Dívida Líquida/EBITDA baixa é positiva.
    df["quality_score"] = (
        0.60 * df["roe_z"]
        - 0.40 * df["net_debt_ebitda_z"]
    )

    # Momentum:
    df["momentum_score"] = df["momentum_12_1_z"]

    # Low risk auxiliar:
    df["low_risk_score"] = (
        0.50 * (-df["vol_12m_z"])
        + 0.50 * df["sortino_12m_z"]
    )

    # Score principal focado em Quality + Momentum
    # Quality é o núcleo; Momentum é o segundo fator; Low Risk ajuda no controle de drawdown.
    df["score"] = (
        0.50 * df["quality_score"]
        + 0.35 * df["momentum_score"]
        + 0.15 * df["low_risk_score"]
    )

    # Opcional: inclui valuation se vocês quiserem defender quality + momentum + valuation.
    if config.use_value_if_available and "earnings_yield_z" in df.columns:
        df["value_score"] = df["earnings_yield_z"]
        df["score"] = (
            0.40 * df["quality_score"]
            + 0.30 * df["momentum_score"]
            + 0.15 * df["low_risk_score"]
            + 0.15 * df["value_score"]
        )

    return df


print("Funções de score carregadas.")

In [ ]:
# ============================================================
# 13. Seleção da carteira
# ============================================================

def apply_weights(portfolio: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    portfolio = portfolio.copy()

    if portfolio.empty:
        return portfolio

    if config.weighting_method == "equal_weight":
        portfolio["weight"] = 1.0 / len(portfolio)

    elif config.weighting_method == "inverse_vol":
        inv_vol = 1.0 / portfolio["vol_12m"].replace(0, np.nan)
        inv_vol = inv_vol.replace([np.inf, -np.inf], np.nan).fillna(0)
        if inv_vol.sum() == 0:
            portfolio["weight"] = 1.0 / len(portfolio)
        else:
            portfolio["weight"] = inv_vol / inv_vol.sum()

    else:
        raise ValueError("weighting_method deve ser 'equal_weight' ou 'inverse_vol'.")

    return portfolio


def select_portfolio_simple(scored: pd.DataFrame, config: BacktestConfig) -> pd.DataFrame:
    df = scored.dropna(subset=["score"]).copy()
    df = df.sort_values("score", ascending=False)

    selected = []
    sector_counts = {}

    max_names_per_sector = int(np.floor(config.n_stocks * config.max_sector_weight))
    max_names_per_sector = max(max_names_per_sector, 1)

    for _, row in df.iterrows():
        sector = row.get("sector", "Unknown")

        if config.use_sector_limit:
            if sector_counts.get(sector, 0) >= max_names_per_sector:
                continue

        selected.append(row)
        sector_counts[sector] = sector_counts.get(sector, 0) + 1

        if len(selected) >= config.n_stocks:
            break

    if not selected:
        return pd.DataFrame()

    portfolio = pd.DataFrame(selected)
    portfolio = apply_weights(portfolio, config)
    return portfolio


def select_portfolio_with_correlation_filter(
    scored: pd.DataFrame,
    returns_matrix: pd.DataFrame,
    asof_date: pd.Timestamp,
    config: BacktestConfig,
) -> pd.DataFrame:
    df = scored.dropna(subset=["score"]).copy()
    df = df.sort_values("score", ascending=False)

    start_date = asof_date - pd.Timedelta(days=365)
    recent_returns = returns_matrix[
        (returns_matrix.index >= start_date) &
        (returns_matrix.index <= asof_date)
    ].copy()

    selected = []
    selected_tickers = []
    sector_counts = {}

    max_names_per_sector = int(np.floor(config.n_stocks * config.max_sector_weight))
    max_names_per_sector = max(max_names_per_sector, 1)

    for _, row in df.iterrows():
        ticker = row["ticker"]
        sector = row.get("sector", "Unknown")

        if ticker not in recent_returns.columns:
            continue

        if config.use_sector_limit:
            if sector_counts.get(sector, 0) >= max_names_per_sector:
                continue

        if selected_tickers:
            corr_data = recent_returns[selected_tickers + [ticker]].dropna(how="all")

            if len(corr_data) >= config.min_lookback_days:
                corr = corr_data.corr()[ticker].drop(ticker)
                max_abs_corr = corr.abs().max()

                if pd.notna(max_abs_corr) and max_abs_corr > config.corr_threshold:
                    continue

        selected.append(row)
        selected_tickers.append(ticker)
        sector_counts[sector] = sector_counts.get(sector, 0) + 1

        if len(selected) >= config.n_stocks:
            break

    if not selected:
        return pd.DataFrame()

    portfolio = pd.DataFrame(selected)
    portfolio = apply_weights(portfolio, config)
    return portfolio


print("Funções de seleção carregadas.")

In [ ]:
# ============================================================
# 14. Backtest engine completo
# ============================================================

def run_backtest(
    prices_prepared: pd.DataFrame,
    fundamentals_prepared: pd.DataFrame,
    market_factors: pd.DataFrame,
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    start_date = pd.Timestamp(config.start_date)
    end_date = pd.Timestamp(config.end_date)

    prices_bt = prices_prepared[
        (prices_prepared["date"] >= start_date) &
        (prices_prepared["date"] <= end_date)
    ].copy()

    market_bt = market_factors[
        (market_factors["date"] >= start_date) &
        (market_factors["date"] <= end_date)
    ].copy()

    returns_matrix = prices_bt.pivot(index="date", columns="ticker", values="return").sort_index()

    rebalance_dates = get_rebalance_dates(prices_bt, config)
    rebalance_dates = rebalance_dates[
        (rebalance_dates >= start_date) &
        (rebalance_dates <= end_date)
    ]

    portfolio_returns = []
    holdings_history = []
    scores_history = []

    previous_weights = pd.Series(dtype=float)

    for i, rebalance_date in enumerate(rebalance_dates):
        if i == len(rebalance_dates) - 1:
            next_rebalance_date = returns_matrix.index.max()
        else:
            next_rebalance_date = rebalance_dates[i + 1]

        market_snapshot = latest_market_snapshot(market_bt, rebalance_date)
        fundamental_snapshot = latest_fundamentals_snapshot(fundamentals_prepared, rebalance_date)

        if market_snapshot.empty or fundamental_snapshot.empty:
            continue

        liquid_universe = select_liquid_universe(market_snapshot, config)

        data = fundamental_snapshot.merge(
            market_snapshot[
                [
                    "ticker",
                    "date",
                    "adtv_12m",
                    "vol_12m",
                    "sharpe_12m",
                    "sortino_12m",
                    "momentum_12_1",
                ]
            ],
            on="ticker",
            how="inner",
        )

        data = data[data["ticker"].isin(liquid_universe)].copy()
        data = apply_basic_filters(data, config)

        if data.empty:
            continue

        scored = compute_quality_momentum_score(data, config)
        scored["rebalance_date"] = rebalance_date
        scores_history.append(scored)

        if config.use_correlation_filter:
            portfolio = select_portfolio_with_correlation_filter(
                scored=scored,
                returns_matrix=returns_matrix,
                asof_date=rebalance_date,
                config=config,
            )
        else:
            portfolio = select_portfolio_simple(scored, config)

        if portfolio.empty:
            continue

        current_weights = portfolio.set_index("ticker")["weight"]

        # Turnover
        all_tickers = previous_weights.index.union(current_weights.index)
        prev = previous_weights.reindex(all_tickers).fillna(0.0)
        curr = current_weights.reindex(all_tickers).fillna(0.0)

        turnover = (curr - prev).abs().sum()
        transaction_cost_value = turnover * config.transaction_cost

        # Retornos até próximo rebalanceamento
        period_returns = returns_matrix[
            (returns_matrix.index > rebalance_date) &
            (returns_matrix.index <= next_rebalance_date)
        ].copy()

        if period_returns.empty:
            continue

        aligned_returns = period_returns.reindex(columns=current_weights.index).fillna(0.0)

        gross_return = aligned_returns.dot(current_weights)
        net_return = gross_return.copy()

        # Custo no primeiro dia do período
        if len(net_return) > 0:
            net_return.iloc[0] -= transaction_cost_value

        temp = pd.DataFrame({
            "date": net_return.index,
            "portfolio_return": net_return.values,
            "gross_return": gross_return.values,
            "turnover": 0.0,
            "transaction_cost": 0.0,
            "rebalance_date": rebalance_date,
        })

        temp.loc[temp.index[0], "turnover"] = turnover
        temp.loc[temp.index[0], "transaction_cost"] = transaction_cost_value

        portfolio_returns.append(temp)

        holdings = portfolio.copy()
        holdings["rebalance_date"] = rebalance_date
        holdings_history.append(holdings)

        previous_weights = current_weights.copy()

    if not portfolio_returns:
        raise ValueError(
            "O backtest não gerou resultados. Verifique dados, datas, colunas e filtros. "
            "Teste reduzir filtros, desligar correlação ou aumentar universo."
        )

    returns_df = pd.concat(portfolio_returns, ignore_index=True).sort_values("date")
    holdings_df = pd.concat(holdings_history, ignore_index=True)
    scores_df = pd.concat(scores_history, ignore_index=True)

    returns_df["cumulative_return"] = (1 + returns_df["portfolio_return"]).cumprod() - 1
    returns_df["gross_cumulative_return"] = (1 + returns_df["gross_return"]).cumprod() - 1
    returns_df["drawdown"] = compute_drawdown(returns_df["portfolio_return"])

    return returns_df, holdings_df, scores_df


returns_df, holdings_df, scores_df = run_backtest(
    prices_prepared=prices_prepared,
    fundamentals_prepared=fundamentals_prepared,
    market_factors=market_factors,
    config=config,
)

print("Backtest concluído.")
display(returns_df.head())
display(holdings_df.head())
display(scores_df.head())

In [ ]:
# ============================================================
# 15. Métricas de performance
# ============================================================

def compute_performance_metrics(
    returns: pd.Series,
    risk_free_daily: float = 0.0,
    periods_per_year: int = 252,
) -> Dict[str, float]:
    r = returns.dropna()
    if len(r) == 0:
        return {}

    total_return = (1 + r).prod() - 1
    ann_return = annualize_return(r, periods_per_year)
    ann_vol = r.std(ddof=0) * np.sqrt(periods_per_year)

    excess = r - risk_free_daily

    sharpe = np.nan
    if excess.std(ddof=0) != 0:
        sharpe = excess.mean() / excess.std(ddof=0) * np.sqrt(periods_per_year)

    downside = excess[excess < 0]
    downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(periods_per_year)

    sortino = np.nan
    if downside_dev != 0 and not np.isnan(downside_dev):
        sortino = excess.mean() * periods_per_year / downside_dev

    dd = compute_drawdown(r)
    max_dd = dd.min()

    calmar = np.nan
    if max_dd != 0 and not np.isnan(max_dd):
        calmar = ann_return / abs(max_dd)

    hit_rate = (r > 0).mean()
    best_day = r.max()
    worst_day = r.min()

    var_95 = r.quantile(0.05)
    cvar_95 = r[r <= var_95].mean()

    return {
        "Retorno acumulado": total_return,
        "Retorno anualizado": ann_return,
        "Volatilidade anualizada": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Calmar": calmar,
        "Máximo drawdown": max_dd,
        "Hit rate diário": hit_rate,
        "Melhor dia": best_day,
        "Pior dia": worst_day,
        "VaR 95% diário": var_95,
        "CVaR 95% diário": cvar_95,
    }


strategy_metrics = compute_performance_metrics(returns_df["portfolio_return"], config.risk_free_daily)
strategy_metrics["Turnover médio por rebalanceamento"] = returns_df.loc[returns_df["turnover"] > 0, "turnover"].mean()
strategy_metrics["Custo total de transação"] = returns_df["transaction_cost"].sum()

metrics_df = pd.DataFrame(strategy_metrics, index=["Estratégia"]).T

display(metrics_df.style.format("{:.4f}"))

In [ ]:
# ============================================================
# 16. Comparação com benchmarks
# ============================================================

def compare_with_benchmarks(
    returns_df: pd.DataFrame,
    benchmarks: Optional[pd.DataFrame],
    config: BacktestConfig,
) -> pd.DataFrame:
    if benchmarks is None:
        return pd.DataFrame()

    df = returns_df[["date", "portfolio_return"]].merge(
        benchmarks,
        on="date",
        how="inner",
    )

    if df.empty:
        return pd.DataFrame()

    rows = {
        "Estratégia": compute_performance_metrics(df["portfolio_return"], config.risk_free_daily)
    }

    benchmark_cols = [c for c in df.columns if c not in ["date", "portfolio_return"]]

    for col in benchmark_cols:
        rows[col] = compute_performance_metrics(df[col], config.risk_free_daily)

    return pd.DataFrame(rows)


def compute_relative_metrics(
    returns_df: pd.DataFrame,
    benchmarks: Optional[pd.DataFrame],
    benchmark_col: str = "ibov_return",
) -> pd.DataFrame:
    if benchmarks is None or benchmark_col not in benchmarks.columns:
        return pd.DataFrame()

    df = returns_df[["date", "portfolio_return"]].merge(
        benchmarks[["date", benchmark_col]],
        on="date",
        how="inner",
    ).dropna()

    if df.empty:
        return pd.DataFrame()

    active = df["portfolio_return"] - df[benchmark_col]
    tracking_error = active.std(ddof=0) * np.sqrt(252)

    information_ratio = np.nan
    if tracking_error != 0:
        information_ratio = active.mean() * 252 / tracking_error

    var_bench = df[benchmark_col].var()
    beta = np.nan

    if var_bench != 0 and not np.isnan(var_bench):
        beta = df[["portfolio_return", benchmark_col]].cov().iloc[0, 1] / var_bench

    alpha_annualized = np.nan
    if not np.isnan(beta):
        alpha_daily = df["portfolio_return"].mean() - beta * df[benchmark_col].mean()
        alpha_annualized = alpha_daily * 252

    out = {
        "Tracking error": tracking_error,
        "Information ratio": information_ratio,
        "Beta": beta,
        "Alpha anualizado": alpha_annualized,
    }

    return pd.DataFrame(out, index=[f"vs {benchmark_col}"]).T


benchmark_comparison = compare_with_benchmarks(returns_df, benchmarks, config)
relative_metrics = compute_relative_metrics(returns_df, benchmarks, benchmark_col="ibov_return")

if not benchmark_comparison.empty:
    display(benchmark_comparison.style.format("{:.4f}"))
else:
    print("Sem comparação com benchmark. Forneça benchmarks_b3.csv para ativar.")

if not relative_metrics.empty:
    display(relative_metrics.style.format("{:.4f}"))

In [ ]:
# ============================================================
# 17. Gráficos principais
# ============================================================

def plot_cumulative_returns(returns_df: pd.DataFrame, benchmarks: Optional[pd.DataFrame] = None) -> None:
    plt.figure(figsize=(12, 6))

    curve = (1 + returns_df.set_index("date")["portfolio_return"]).cumprod()
    plt.plot(curve.index, curve.values, label="Estratégia Quality + Momentum")

    if benchmarks is not None:
        b = benchmarks.copy().set_index("date")
        b = b.loc[(b.index >= returns_df["date"].min()) & (b.index <= returns_df["date"].max())]

        for col in b.columns:
            bench_curve = (1 + b[col].dropna()).cumprod()
            plt.plot(bench_curve.index, bench_curve.values, label=col)

    plt.title("Retorno acumulado — Quality + Momentum B3")
    plt.xlabel("Data")
    plt.ylabel("Crescimento de R$1")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_drawdown(returns_df: pd.DataFrame) -> None:
    plt.figure(figsize=(12, 6))
    plt.plot(returns_df["date"], returns_df["drawdown"], label="Drawdown")
    plt.title("Drawdown — Estratégia Quality + Momentum B3")
    plt.xlabel("Data")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_rolling_sharpe(returns_df: pd.DataFrame, window: int = 126) -> None:
    r = returns_df.set_index("date")["portfolio_return"]
    rolling_sharpe = (
        r.rolling(window)
        .apply(lambda x: sharpe_ratio(pd.Series(x), config.risk_free_daily), raw=False)
    )

    plt.figure(figsize=(12, 6))
    plt.plot(rolling_sharpe.index, rolling_sharpe.values, label=f"Sharpe móvel {window} dias")
    plt.title("Sharpe móvel")
    plt.xlabel("Data")
    plt.ylabel("Sharpe anualizado")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_rolling_sortino(returns_df: pd.DataFrame, window: int = 126) -> None:
    r = returns_df.set_index("date")["portfolio_return"]
    rolling_sortino = (
        r.rolling(window)
        .apply(lambda x: sortino_ratio(pd.Series(x), config.risk_free_daily), raw=False)
    )

    plt.figure(figsize=(12, 6))
    plt.plot(rolling_sortino.index, rolling_sortino.values, label=f"Sortino móvel {window} dias")
    plt.title("Sortino móvel")
    plt.xlabel("Data")
    plt.ylabel("Sortino anualizado")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


plot_cumulative_returns(returns_df, benchmarks)
plot_drawdown(returns_df)
plot_rolling_sharpe(returns_df)
plot_rolling_sortino(returns_df)

In [ ]:
# ============================================================
# 18. Gráficos adicionais: turnover, pesos setoriais, fatores
# ============================================================

def plot_turnover(returns_df: pd.DataFrame) -> None:
    turnover = returns_df[returns_df["turnover"] > 0][["date", "turnover"]]

    plt.figure(figsize=(12, 6))
    plt.bar(turnover["date"], turnover["turnover"], width=20)
    plt.title("Turnover por rebalanceamento")
    plt.xlabel("Data")
    plt.ylabel("Turnover")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_sector_weights(holdings_df: pd.DataFrame) -> None:
    if "sector" not in holdings_df.columns:
        print("Sem coluna sector.")
        return

    sector_weights = holdings_df.pivot_table(
        index="rebalance_date",
        columns="sector",
        values="weight",
        aggfunc="sum",
    ).fillna(0)

    ax = sector_weights.plot(kind="area", stacked=True, figsize=(12, 6))
    ax.set_title("Composição setorial da carteira")
    ax.set_xlabel("Data de rebalanceamento")
    ax.set_ylabel("Peso")
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()


def plot_factor_exposures(holdings_df: pd.DataFrame) -> None:
    factor_cols = [
        "quality_score",
        "momentum_score",
        "low_risk_score",
        "score",
    ]

    available = [c for c in factor_cols if c in holdings_df.columns]

    if not available:
        print("Sem fatores para plotar.")
        return

    exposures = holdings_df.groupby("rebalance_date")[available].mean()

    plt.figure(figsize=(12, 6))
    for col in available:
        plt.plot(exposures.index, exposures[col], label=col)

    plt.title("Exposição média aos fatores na carteira")
    plt.xlabel("Data de rebalanceamento")
    plt.ylabel("Score médio")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


plot_turnover(returns_df)
plot_sector_weights(holdings_df)
plot_factor_exposures(holdings_df)

In [ ]:
# ============================================================
# 19. Tabela das carteiras por rebalanceamento
# ============================================================

cols_to_show = [
    "rebalance_date",
    "ticker",
    "sector",
    "weight",
    "score",
    "quality_score",
    "momentum_score",
    "low_risk_score",
    "roe",
    "net_debt_ebitda",
    "momentum_12_1",
    "vol_12m",
    "sortino_12m",
]

cols_to_show = [c for c in cols_to_show if c in holdings_df.columns]

display(
    holdings_df[cols_to_show]
    .sort_values(["rebalance_date", "score"], ascending=[True, False])
    .head(60)
)

In [ ]:
# ============================================================
# 20. Performance por ano
# ============================================================

def yearly_performance(returns_df: pd.DataFrame) -> pd.DataFrame:
    df = returns_df.copy()
    df["year"] = df["date"].dt.year

    rows = []

    for year, g in df.groupby("year"):
        r = g["portfolio_return"]
        metrics = compute_performance_metrics(r, config.risk_free_daily)
        rows.append({
            "Ano": year,
            "Retorno": metrics.get("Retorno acumulado", np.nan),
            "Volatilidade anualizada": metrics.get("Volatilidade anualizada", np.nan),
            "Sharpe": metrics.get("Sharpe", np.nan),
            "Sortino": metrics.get("Sortino", np.nan),
            "Max Drawdown": metrics.get("Máximo drawdown", np.nan),
        })

    return pd.DataFrame(rows)


yearly_df = yearly_performance(returns_df)
display(yearly_df.style.format({
    "Retorno": "{:.2%}",
    "Volatilidade anualizada": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
}))

In [ ]:
# ============================================================
# 21. Testes de robustez simples
# ============================================================

def run_robustness_tests(
    prices_prepared: pd.DataFrame,
    fundamentals_prepared: pd.DataFrame,
    market_factors: pd.DataFrame,
    base_config: BacktestConfig,
) -> pd.DataFrame:
    tests = []

    scenarios = [
        {"name": "Base", "n_stocks": 20, "use_correlation_filter": True, "weighting_method": "equal_weight"},
        {"name": "Top 10", "n_stocks": 10, "use_correlation_filter": True, "weighting_method": "equal_weight"},
        {"name": "Top 30", "n_stocks": 30, "use_correlation_filter": True, "weighting_method": "equal_weight"},
        {"name": "Sem filtro correlação", "n_stocks": 20, "use_correlation_filter": False, "weighting_method": "equal_weight"},
        {"name": "Inverse Vol", "n_stocks": 20, "use_correlation_filter": True, "weighting_method": "inverse_vol"},
    ]

    for s in scenarios:
        cfg = BacktestConfig(**base_config.__dict__)
        cfg.n_stocks = s["n_stocks"]
        cfg.use_correlation_filter = s["use_correlation_filter"]
        cfg.weighting_method = s["weighting_method"]

        try:
            r, h, sc = run_backtest(prices_prepared, fundamentals_prepared, market_factors, cfg)
            m = compute_performance_metrics(r["portfolio_return"], cfg.risk_free_daily)
            tests.append({
                "Cenário": s["name"],
                "Retorno anualizado": m.get("Retorno anualizado", np.nan),
                "Vol anualizada": m.get("Volatilidade anualizada", np.nan),
                "Sharpe": m.get("Sharpe", np.nan),
                "Sortino": m.get("Sortino", np.nan),
                "Max Drawdown": m.get("Máximo drawdown", np.nan),
                "Retorno acumulado": m.get("Retorno acumulado", np.nan),
            })
        except Exception as e:
            tests.append({
                "Cenário": s["name"],
                "Erro": str(e),
            })

    return pd.DataFrame(tests)


robustness_df = run_robustness_tests(
    prices_prepared=prices_prepared,
    fundamentals_prepared=fundamentals_prepared,
    market_factors=market_factors,
    base_config=config,
)

display(robustness_df)

In [ ]:
# ============================================================
# 22. Exportação dos resultados
# ============================================================

def export_all_results(
    returns_df: pd.DataFrame,
    holdings_df: pd.DataFrame,
    scores_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    yearly_df: pd.DataFrame,
    robustness_df: pd.DataFrame,
    benchmark_comparison: pd.DataFrame,
    relative_metrics: pd.DataFrame,
    output_dir: Path,
) -> None:
    output_dir.mkdir(exist_ok=True)

    returns_df.to_csv(output_dir / "strategy_returns.csv", index=False)
    holdings_df.to_csv(output_dir / "holdings_history.csv", index=False)
    scores_df.to_csv(output_dir / "scores_history.csv", index=False)
    metrics_df.to_csv(output_dir / "strategy_metrics.csv")
    yearly_df.to_csv(output_dir / "yearly_performance.csv", index=False)
    robustness_df.to_csv(output_dir / "robustness_tests.csv", index=False)

    if not benchmark_comparison.empty:
        benchmark_comparison.to_csv(output_dir / "benchmark_comparison.csv")

    if not relative_metrics.empty:
        relative_metrics.to_csv(output_dir / "relative_metrics.csv")

    excel_path = output_dir / "backtest_quality_momentum_results.xlsx"

    with pd.ExcelWriter(excel_path) as writer:
        returns_df.to_excel(writer, sheet_name="returns", index=False)
        holdings_df.to_excel(writer, sheet_name="holdings", index=False)
        scores_df.to_excel(writer, sheet_name="scores", index=False)
        metrics_df.to_excel(writer, sheet_name="metrics")
        yearly_df.to_excel(writer, sheet_name="yearly", index=False)
        robustness_df.to_excel(writer, sheet_name="robustness", index=False)
        if not benchmark_comparison.empty:
            benchmark_comparison.to_excel(writer, sheet_name="benchmarks")
        if not relative_metrics.empty:
            relative_metrics.to_excel(writer, sheet_name="relative")

    print(f"Resultados exportados para: {output_dir.resolve()}")
    print(f"Excel consolidado: {excel_path.resolve()}")


export_all_results(
    returns_df=returns_df,
    holdings_df=holdings_df,
    scores_df=scores_df,
    metrics_df=metrics_df,
    yearly_df=yearly_df,
    robustness_df=robustness_df,
    benchmark_comparison=benchmark_comparison,
    relative_metrics=relative_metrics,
    output_dir=OUTPUT_DIR,
)

# 23. Como interpretar os resultados na apresentação

Use esta estrutura:

## Tese

A carteira combina **Quality** e **Momentum**:

- Quality busca empresas com fundamentos sólidos: ROE alto e baixa alavancagem.
- Momentum captura persistência de tendência de preço.
- Low Risk entra como controle auxiliar para reduzir drawdowns.

## Metodologia

- Universo: 100 ações mais líquidas da B3.
- Período: 2021–2026.
- Fonte: Economática.
- Rebalanceamento: trimestral.
- Carteira: 20 ações long-only.
- Pesos: iguais.
- Custos: 20 bps por turnover.
- Score:

```text
Score = 50% Quality + 35% Momentum + 15% Low Risk
```

com:

```text
Quality = 60% z(ROE) - 40% z(Dívida Líquida/EBITDA)
Momentum = z(retorno 12-1 meses)
Low Risk = 50% [-z(volatilidade 12M)] + 50% z(Sortino 12M)
```

## O que mostrar

1. Retorno acumulado.
2. Drawdown.
3. Sharpe e Sortino.
4. Performance anual.
5. Composição setorial.
6. Testes de robustez.
7. Limitações.

## Limitações importantes

- Possível survivorship bias se o universo vier de ações atuais.
- Dependência da qualidade dos dados do Economática.
- Custos e impostos simplificados.
- Liquidez aproximada por volume financeiro médio.
- Defasagem de 3 meses nos fundamentos para evitar look-ahead bias.